# Your First RAG Application

In this notebook, we'll walk you through each of the components that are involved in a simple RAG application.

We won't be leveraging any fancy tools, just the OpenAI Python SDK, Numpy, and some classic Python.

> NOTE: This was done with Python 3.11.4.

> NOTE: There might be [compatibility issues](https://github.com/wandb/wandb/issues/7683) if you're on NVIDIA driver >552.44 As an interim solution - you can rollback your drivers to the 552.44.

## Table of Contents:

- Task 1: Imports and Utilities
- Task 2: Documents
- Task 3: Embeddings and Vectors
- Task 4: Prompts
- Task 5: Retrieval Augmented Generation
  - 🚧 Activity #1: Augment RAG

Let's look at a rather complicated looking visual representation of a basic RAG application.

<img src="https://i.imgur.com/vD8b016.png" />

## Task 1: Imports and Utility

We're just doing some imports and enabling `async` to work within the Jupyter environment here, nothing too crazy!

In [5]:
from aimakerspace.text_utils import TextFileLoader, CharacterTextSplitter
from aimakerspace.vectordatabase import VectorDatabase
from aimakerspace.openai_utils.prompts import SystemRolePrompt, UserRolePrompt
from aimakerspace.openai_utils.chatmodel import ChatOpenAI
import asyncio

In [6]:
import nest_asyncio
nest_asyncio.apply()

## Task 2: Documents

We'll be concerning ourselves with this part of the flow in the following section:

<img src="https://i.imgur.com/jTm9gjk.png" />

### Loading Source Documents

So, first things first, we need some documents to work with.

While we could work directly with the `.txt` files (or whatever file-types you wanted to extend this to) we can instead do some batch processing of those documents at the beginning in order to store them in a more machine compatible format.

In this case, we're going to parse our text file into a single document in memory.

Let's look at the relevant bits of the `TextFileLoader` class:

```python
def load_file(self):
        with open(self.path, "r", encoding=self.encoding) as f:
            self.documents.append(f.read())
```

We're simply loading the document using the built in `open` method, and storing that output in our `self.documents` list.

> NOTE: We're using blogs from PMarca (Marc Andreessen) as our sample data. This data is largely irrelevant as we want to focus on the mechanisms of RAG, which includes out data's shape and quality - but not specifically what the contents of the data are. 


In [7]:
# Load documents from PDF file (the pre-existing txt file was converted 
# to a pdf). The TextFileLoader class was also updated to support pdf files:w
#  
pdf_loader = TextFileLoader("data/PMarcaBlogs.pdf")
documents = pdf_loader.load_documents()
len(documents)

1

In [8]:
print(documents[0][:100])

The  Pmarca  Blog  Archives  (select  posts  from  2007-2009)  Marc  Andreessen  copyright:  Andrees


### Splitting Text Into Chunks

As we can see, there is one massive document.

We'll want to chunk the document into smaller parts so it's easier to pass the most relevant snippets to the LLM.

There is no fixed way to split/chunk documents - and you'll need to rely on some intuition as well as knowing your data *very* well in order to build the most robust system.

For this toy example, we'll just split blindly on length.

>There's an opportunity to clear up some terminology here, for this course we will be stick to the following:
>
>- "source documents" : The `.txt`, `.pdf`, `.html`, ..., files that make up the files and information we start with in its raw format
>- "document(s)" : single (or more) text object(s)
>- "corpus" : the combination of all of our documents

As you can imagine (though it's not specifically true in this toy example) the idea of splitting documents is to break them into managable sized chunks that retain the most relevant local context.

In [9]:
# Split documents with better chunking parameters
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_documents = text_splitter.split_texts(documents)
len(split_documents)

441

Let's take a look at some of the documents we've managed to split.

In [10]:
split_documents[0:1]

['The  Pmarca  Blog  Archives  (select  posts  from  2007-2009)  Marc  Andreessen  copyright:  Andreessen  Horowitz  cover  design:  Jessica  Hagy  produced  using:  Pressbooks  Contents  THE  PMARCA  GUIDE  TO  STARTUPS  Part  1:  Why  not  to  do  a  startup  2  Part  2:  When  the  VCs  say  "no"  10  Part  3:  "But  I  don\'t  know  any  VCs!"  18  Part  4:  The  only  thing  that  matters  25  Part  5:  The  Moby  Dick  theory  of  big  companies  33  Part  6:  How  much  funding  is  too  little?  Too  much?  41  Part  7:  Why  a  startup\'s  initial  business  plan  doesn\'t  matter  that  much  49  THE  PMARCA  GUIDE  TO  HIRING  Part  8:  Hiring,  managing,  promoting,  and  Dring  executives  54  Part  9:  How  to  hire  a  professional  CEO  68  How  to  hire  the  best  people  you\'ve  ever  worked  with  69  THE  PMARCA  GUIDE  TO  BIG  COMPANIES  Part  1:  Turnaround!  82  Part  2:  Retaining  great  people  86  THE  PMARCA  GUIDE  TO  CAREER,  PRODUCTIVITY,  AND  SOME  

## Task 3: Embeddings and Vectors

Next, we have to convert our corpus into a "machine readable" format as we explored in the Embedding Primer notebook.

Today, we're going to talk about the actual process of creating, and then storing, these embeddings, and how we can leverage that to intelligently add context to our queries.

### OpenAI API Key

In order to access OpenAI's APIs, we'll need to provide our OpenAI API Key!

You can work through the folder "OpenAI API Key Setup" for more information on this process if you don't already have an API Key!

In [11]:
import os
import openai
from getpass import getpass

openai.api_key = getpass("OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai.api_key

### Vector Database

Let's set up our vector database to hold all our documents and their embeddings!

While this is all baked into 1 call - we can look at some of the code that powers this process to get a better understanding:

Let's look at our `VectorDatabase().__init__()`:

```python
def __init__(self, embedding_model: EmbeddingModel = None):
        self.vectors = defaultdict(np.array)
        self.embedding_model = embedding_model or EmbeddingModel()
```

As you can see - our vectors are merely stored as a dictionary of `np.array` objects.

Secondly, our `VectorDatabase()` has a default `EmbeddingModel()` which is a wrapper for OpenAI's `text-embedding-3-small` model.

> **Quick Info About `text-embedding-3-small`**:
> - It has a context window of **8191** tokens
> - It returns vectors with dimension **1536**

#### ❓Question #1:

The default embedding dimension of `text-embedding-3-small` is 1536, as noted above. 

1. Is there any way to modify this dimension? - _No..the dimensions are fixed for a given model, and depending on the size of the model may change across different models. Though the dimensions are fixed, the openai api allows requesting a subset of the dimensions (e.g. using fewer of the numbers in the vector form the embedding._
2. What technique does OpenAI use to achieve this? - _According to their documentation OpenAI has an optional input for dimensions. This is a request for the desired number of dimensions from the total number the embedding model provides.  The api truncates the vector but does not do any additional processing._

> NOTE: Check out this [API documentation](https://platform.openai.com/docs/api-reference/embeddings/create) for the answer to question #1, and [this documentation](https://platform.openai.com/docs/guides/embeddings/use-cases) for an answer to question #2!

We can call the `async_get_embeddings` method of our `EmbeddingModel()` on a list of `str` and receive a list of `float` back!

```python
async def async_get_embeddings(self, list_of_text: List[str]) -> List[List[float]]:
        return await aget_embeddings(
            list_of_text=list_of_text, engine=self.embeddings_model_name
        )
```

We cast those to `np.array` when we build our `VectorDatabase()`:

```python
async def abuild_from_list(self, list_of_text: List[str]) -> "VectorDatabase":
        embeddings = await self.embedding_model.async_get_embeddings(list_of_text)
        for text, embedding in zip(list_of_text, embeddings):
            self.insert(text, np.array(embedding))
        return self
```

And that's all we need to do!

In [12]:
import asyncio
from aimakerspace.vectordatabase import VectorDatabase

vector_db = VectorDatabase()
vector_db = asyncio.run(vector_db.abuild_from_list(split_documents))








# 

#### ❓Question #2:

What are the benefits of using an `async` approach to collecting our embeddings? 
_Async vs. Sync:
- Synchronous (sync): Each API call waits for the previous one to finish before starting the next. 
- Asynchronous (async): Multiple API calls can be sent out at the same time, without waiting for each one to finish before starting the next. 

If 10 documents have 50 chunks that may be manageable, but an enterprise that may do embeddings on a large history of documents may find this unsustainable without using non-blocking requests for embeddings. This may be even more significant if there's analysis on data that continually updated at scale... like analysis for customer support emails or recordings from technical support calls. _

> NOTE: Determining the core difference between `async` and `sync` will be useful! If you get stuck - ask ChatGPT!

So, to review what we've done so far in natural language:

1. We load source documents
2. We split those source documents into smaller chunks (documents)
3. We send each of those documents to the `text-embedding-3-small` OpenAI API endpoint
4. We store each of the text representations with the vector representations as keys/values in a dictionary

### Semantic Similarity

The next step is to be able to query our `VectorDatabase()` with a `str` and have it return to us vectors and text that is most relevant from our corpus.

We're going to use the following process to achieve this in our toy example:

1. We need to embed our query with the same `EmbeddingModel()` as we used to construct our `VectorDatabase()`
2. We loop through every vector in our `VectorDatabase()` and use a distance measure to compare how related they are
3. We return a list of the top `k` closest vectors, with their text representations

There's some very heavy optimization that can be done at each of these steps - but let's just focus on the basic pattern in this notebook.

> We are using [cosine similarity](https://www.engati.com/glossary/cosine-similarity) as a distance metric in this example - but there are many many distance metrics you could use - like [these](https://flavien-vidal.medium.com/similarity-distances-for-natural-language-processing-16f63cd5ba55)

> We are using a rather inefficient way of calculating relative distance between the query vector and all other vectors - there are more advanced approaches that are much more efficient, like [ANN](https://towardsdatascience.com/comprehensive-guide-to-approximate-nearest-neighbors-algorithms-8b94f057d6b6)

In [13]:
vector_db.search_by_text("What is the Michael Eisner Memorial Weak Executive Problem?", k=3)

[('n’t  want  your  key  executives  focused  on  selling  the  company  —  unless  of  course  you  want  them  focused  on  selling  the  company.  Make  your  acceleration  decisions  accordingly.  Seventh,  when  hiring  the  executive  to  run  your  former  specialty,  be  \ncareful  you  don’t  hire  someone  weak  on  purpose.  This  sounds  silly,  but  you  wouldn’t  believe  how  oaen  it  happens.  The  CEO  who  used  to  be  a  product  manager  who  has  a  weak  product  management  executive.  The  CEO  who  used  to  be  in  sales  who  has  a  weak  sales  executive.  The  CEO  who  used  to  be  in  marketing  who  has  a  weak  marketing  executive.  I  call  this  the  “Michael  Eisner  Memorial  Weak  Executive  Problem”  —  aaer  the  CEO  of  Disney  \nwho\n \nhad\n \npreviously\n \nbeen\n \na\n \nbrilliant\n \nTV\n \nnetwork\n \nexecutive.\n \nWhen\n \nhe\n \nbought\n \nABC\n \nat\n \nDisney,\n \nit\n promptly  fell  to  fourth  place.  His  response?  “If  I 

## Task 4: Prompts

In the following section, we'll be looking at the role of prompts - and how they help us to guide our application in the right direction.

In this notebook, we're going to rely on the idea of "zero-shot in-context learning".

This is a lot of words to say: "We will ask it to perform our desired task in the prompt, and provide no examples."

### XYZRolePrompt

Before we do that, let's stop and think a bit about how OpenAI's chat models work.

We know they have roles - as is indicated in the following API [documentation](https://platform.openai.com/docs/api-reference/chat/create#chat/create-messages)

There are three roles, and they function as follows (taken directly from [OpenAI](https://platform.openai.com/docs/guides/gpt/chat-completions-api)):

- `{"role" : "system"}` : The system message helps set the behavior of the assistant. For example, you can modify the personality of the assistant or provide specific instructions about how it should behave throughout the conversation. However note that the system message is optional and the model’s behavior without a system message is likely to be similar to using a generic message such as "You are a helpful assistant."
- `{"role" : "user"}` : The user messages provide requests or comments for the assistant to respond to.
- `{"role" : "assistant"}` : Assistant messages store previous assistant responses, but can also be written by you to give examples of desired behavior.

The main idea is this:

1. You start with a system message that outlines how the LLM should respond, what kind of behaviours you can expect from it, and more
2. Then, you can provide a few examples in the form of "assistant"/"user" pairs
3. Then, you prompt the model with the true "user" message.

In this example, we'll be forgoing the 2nd step for simplicities sake.

#### Utility Functions

You'll notice that we're using some utility functions from the `aimakerspace` module - let's take a peek at these and see what they're doing!

##### XYZRolePrompt

Here we have our `system`, `user`, and `assistant` role prompts.

Let's take a peek at what they look like:

```python
class BasePrompt:
    def __init__(self, prompt):
        """
        Initializes the BasePrompt object with a prompt template.

        :param prompt: A string that can contain placeholders within curly braces
        """
        self.prompt = prompt
        self._pattern = re.compile(r"\{([^}]+)\}")

    def format_prompt(self, **kwargs):
        """
        Formats the prompt string using the keyword arguments provided.

        :param kwargs: The values to substitute into the prompt string
        :return: The formatted prompt string
        """
        matches = self._pattern.findall(self.prompt)
        return self.prompt.format(**{match: kwargs.get(match, "") for match in matches})

    def get_input_variables(self):
        """
        Gets the list of input variable names from the prompt string.

        :return: List of input variable names
        """
        return self._pattern.findall(self.prompt)
```

Then we have our `RolePrompt` which laser focuses us on the role pattern found in most API endpoints for LLMs.

```python
class RolePrompt(BasePrompt):
    def __init__(self, prompt, role: str):
        """
        Initializes the RolePrompt object with a prompt template and a role.

        :param prompt: A string that can contain placeholders within curly braces
        :param role: The role for the message ('system', 'user', or 'assistant')
        """
        super().__init__(prompt)
        self.role = role

    def create_message(self, **kwargs):
        """
        Creates a message dictionary with a role and a formatted message.

        :param kwargs: The values to substitute into the prompt string
        :return: Dictionary containing the role and the formatted message
        """
        return {"role": self.role, "content": self.format_prompt(**kwargs)}
```

We'll look at how the `SystemRolePrompt` is constructed to get a better idea of how that extension works:

```python
class SystemRolePrompt(RolePrompt):
    def __init__(self, prompt: str):
        super().__init__(prompt, "system")
```

That pattern is repeated for our `UserRolePrompt` and our `AssistantRolePrompt` as well.

##### ChatOpenAI

Next we have our model, which is converted to a format analagous to libraries like LangChain and LlamaIndex.

Let's take a peek at how that is constructed:

```python
class ChatOpenAI:
    def __init__(self, model_name: str = "gpt-4o-mini"):
        self.model_name = model_name
        self.openai_api_key = os.getenv("OPENAI_API_KEY")
        if self.openai_api_key is None:
            raise ValueError("OPENAI_API_KEY is not set")

    def run(self, messages, text_only: bool = True):
        if not isinstance(messages, list):
            raise ValueError("messages must be a list")

        openai.api_key = self.openai_api_key
        response = openai.ChatCompletion.create(
            model=self.model_name, messages=messages
        )

        if text_only:
            return response.choices[0].message.content

        return response
```

#### ❓ Question #3:

When calling the OpenAI API - are there any ways we can achieve more reproducible outputs?

Yes... several options for achieving reproducible outputs include:
- Consistent prompt formatting - stay consistent with formatting such as whitespace, punctuation, system vs. user message boundaries
- pinning applications to specific model snapshots - consistent request to the same model snapshot should provide similar results
- Temperature - The temperature parameter controls randomness. Lowering the value reduces randomness, however to make results more reproduciable it may be good to set temperature to a fixed value (e.g. 0)

> NOTE: Check out [this section](https://platform.openai.com/docs/guides/text-generation/) of the OpenAI documentation for the answer!

### Creating and Prompting OpenAI's `gpt-4o-mini`!

Let's tie all these together and use it to prompt `gpt-4o-mini`!

In [14]:
from aimakerspace.openai_utils.prompts import (
    UserRolePrompt,
    SystemRolePrompt,
    AssistantRolePrompt,
)

from aimakerspace.openai_utils.chatmodel import ChatOpenAI

chat_openai = ChatOpenAI()
user_prompt_template = "{content}"
user_role_prompt = UserRolePrompt(user_prompt_template)
system_prompt_template = (
    "You are an expert in {expertise}, you always answer in a kind way."
)
system_role_prompt = SystemRolePrompt(system_prompt_template)

messages = [
    system_role_prompt.create_message(expertise="Python"),
    user_role_prompt.create_message(
        content="What is the best way to write a loop?"
    ),
]

response = chat_openai.run(messages)

In [15]:
print(response)

The best way to write a loop in Python largely depends on what you're trying to achieve. Here are some common and effective ways to write loops:

### For Loop
If you're iterating over a sequence (like a list, tuple, or string), the `for` loop is usually the best option:

```python
for item in iterable:
    # Perform actions using item
```

**Example:**
```python
fruits = ['apple', 'banana', 'cherry']
for fruit in fruits:
    print(fruit)
```

### While Loop
If you need to loop until a certain condition is met, the `while` loop is appropriate:

```python
while condition:
    # Perform actions
```

**Example:**
```python
count = 0
while count < 5:
    print(count)
    count += 1
```

### Looping through a Range
If you want to iterate a specific number of times, you can use `range()` with a `for` loop:

```python
for i in range(5):  # Loop for 5 iterations, index 0 to 4
    print(i)
```

### List Comprehensions
For concise and efficient loops, especially when creating lists, you can use l

## Task 5: Retrieval Augmented Generation

Now we can create a RAG prompt - which will help our system behave in a way that makes sense!

There is much you could do here, many tweaks and improvements to be made!

In [16]:
RAG_SYSTEM_TEMPLATE = """You are a knowledgeable assistant that answers questions based strictly on provided context.

Instructions:
- Only answer questions using information from the provided context
- If the context doesn't contain relevant information, respond with "I don't know"
- Be accurate and cite specific parts of the context when possible
- Keep responses {response_style} and {response_length}
- Only use the provided context. Do not use external knowledge.
- Only provide answers when you are confident the context supports your response."""

RAG_USER_TEMPLATE = """Context Information:
{context}

Number of relevant sources found: {context_count}
{similarity_scores}

Question: {user_query}

Please provide your answer based solely on the context above."""

rag_system_prompt = SystemRolePrompt(
    RAG_SYSTEM_TEMPLATE,
    strict=True,
    defaults={
        "response_style": "concise",
        "response_length": "brief"
    }
)

rag_user_prompt = UserRolePrompt(
    RAG_USER_TEMPLATE,
    strict=True,
    defaults={
        "context_count": "",
        "similarity_scores": ""
    }
)

Now we can create our pipeline!

In [17]:
class RetrievalAugmentedQAPipeline:
    def __init__(self, llm: ChatOpenAI(), vector_db_retriever: VectorDatabase, 
                 response_style: str = "detailed", include_scores: bool = False) -> None:
        self.llm = llm
        self.vector_db_retriever = vector_db_retriever
        self.response_style = response_style
        self.include_scores = include_scores

    def run_pipeline(self, user_query: str, k: int = 4, **system_kwargs) -> dict:
        # Retrieve relevant contexts
        context_list = self.vector_db_retriever.search_by_text(user_query, k=k)
        
        context_prompt = ""
        similarity_scores = []
        
        for i, (context, score) in enumerate(context_list, 1):
            context_prompt += f"[Source {i}]: {context}\n\n"
            similarity_scores.append(f"Source {i}: {score:.3f}")
        
        # Create system message with parameters
        system_params = {
            "response_style": self.response_style,
            "response_length": system_kwargs.get("response_length", "detailed")
        }
        
        formatted_system_prompt = rag_system_prompt.create_message(**system_params)
        
        user_params = {
            "user_query": user_query,
            "context": context_prompt.strip(),
            "context_count": len(context_list),
            "similarity_scores": f"Relevance scores: {', '.join(similarity_scores)}" if self.include_scores else ""
        }
        
        formatted_user_prompt = rag_user_prompt.create_message(**user_params)

        return {
            "response": self.llm.run([formatted_system_prompt, formatted_user_prompt]), 
            "context": context_list,
            "context_count": len(context_list),
            "similarity_scores": similarity_scores if self.include_scores else None,
            "prompts_used": {
                "system": formatted_system_prompt,
                "user": formatted_user_prompt
            }
        }

In [18]:
rag_pipeline = RetrievalAugmentedQAPipeline(
    vector_db_retriever=vector_db,
    llm=chat_openai,
    response_style="detailed",
    include_scores=True
)

result = rag_pipeline.run_pipeline(
    "What is the 'Michael Eisner Memorial Weak Executive Problem'?",
    k=3,
    response_length="comprehensive"
)

print(f"Response: {result['response']}")
print(f"\nContext Count: {result['context_count']}")
print(f"Similarity Scores: {result['similarity_scores']}")


Response: The 'Michael Eisner Memorial Weak Executive Problem' refers to the tendency for CEOs, particularly those who have previously excelled in one area (such as product management, sales, or marketing), to hire executives who are weak in the same area that the CEO was once strong. This phenomenon is named after Michael Eisner, the former CEO of Disney, who, despite being a successful TV network executive, saw a decline in ABC's performance after acquiring it, bringing attention to the potential pitfalls of hiring underperforming executives based on prior strengths. This problem highlights the importance of careful hiring choices to avoid filling leadership positions with individuals who may not be adequately skilled or competent.

Context Count: 3
Similarity Scores: ['Source 1: 0.606', 'Source 2: 0.476', 'Source 3: 0.470']


#### ❓ Question #4:

What prompting strategies could you use to make the LLM have a more thoughtful, detailed response? - _The strategy is called "Chain of Thought". Using this strategy the LLM is instructed to  "think step by step" or "explain your reasoning". The model generates intermediate reasoning steps leading to more thoughtful and detailed answers._

> NOTE: You can look through ["Accessing GPT-3.5-turbo Like a Developer"](https://colab.research.google.com/drive/1mOzbgf4a2SP5qQj33ZxTz2a01-5eXqk2?usp=sharing) for an answer to this question if you get stuck!

### 🏗️ Activity #1:

Enhance your RAG application in some way! 

Suggestions are: 

- Allow it to work with PDF files
- Implement a new distance metric
- Add metadata support to the vector database

While these are suggestions, you should feel free to make whatever augmentations you desire! 

> NOTE: These additions might require you to work within the `aimakerspace` library - that's expected!

> NOTE: If you're not sure where to start - ask Cursor (CMD/CTRL+L) to guide you through the changes!

In [19]:
# Enhanced RAG Pipeline with PDF Support
# The pipeline now supports PDF files and has improved prompts

print("🎉 Your RAG pipeline now supports PDF files!")
print("📄 To use with other PDF files:")
print("   1. Place your PDF files in the data/ directory")
print("   2. Use TextFileLoader('data/your_file.pdf')")
print("   3. The pipeline will automatically extract text and create embeddings")

# Test with a different query
test_query = "What are the key principles of startup management?"
result = rag_pipeline.run_pipeline(test_query, k=2)

print(f"\nTest Query: {test_query}")
print(f"Response: {result['response']}")

🎉 Your RAG pipeline now supports PDF files!
📄 To use with other PDF files:
   1. Place your PDF files in the data/ directory
   2. Use TextFileLoader('data/your_file.pdf')
   3. The pipeline will automatically extract text and create embeddings

Test Query: What are the key principles of startup management?
Response: The key principles of startup management based on the provided context include:

1. **Hiring Strategy**: Startups should be cautious about hiring executives too early in the process. It is advisable to first assess whether an executive is genuinely needed for a function. There is a tendency for well-funded startups to hire executives prematurely before achieving product/market fit. Instead, having a highly motivated manager or director may be more beneficial than bringing in a costly executive who may not contribute significantly to the team (Source 1).

2. **Control Over Destiny**: One of the fundamental attractions of managing a startup is the opportunity for individuals